# KPI Goal Cascade Workflow

This notebook walks through the full pipeline to cascade goals for a KPI.

**Steps:**
1. Query data from Teradata (or use existing cache)
2. Design a hierarchy
3. Define goals
4. Run the cascade
5. View results in the Streamlit app

---

## Step 1: Pull Data from Teradata

Run the cell below to query Teradata and save `teradata_cache.parquet`.

**Edit the query** to change the KPI, date range, or add new dimensions.
Any column you add here (besides `num`, `den`, `Yr_Nb`) will automatically
appear as an available dimension in the hierarchy editor.

**Skip this step** if `teradata_cache.parquet` already exists and is up to date.

In [ ]:
import teradatasql
import pandas as pd
import getpass as gp
import os
from pathlib import Path

# --- Connection ---
TD_HOST = os.getenv('TD_HOST', 'dwprod')
TD_USER = os.getenv('TD_USER') or input('Teradata username: ')
TD_PASS = os.getenv('TD_PASS') or gp.getpass('Teradata password: ')
TD_LOGMECH = os.getenv('TD_LOGMECH', 'LDAP')

# --- Query ---
# Edit this query to change KPI, date range, or dimensions.
# Rules:
#   - Must include 'num' (KPI numerator) and 'den' (denominator)
#   - All other columns become available dimensions for hierarchy building
#   - Column aliases become the dimension names (e.g. ml_dc_1, station, fleet)

query = """
select
    Yr_Nb,
    Mo_Nb,
    'sys' as sys,
    ML_DC_Cd as ml_dc_1,
    Ownr_Crr_Cd as ml_dc_2,
    Dom_Intl_Cd as dom_int,
    Schd_Orig_Stn_Cd as station,
    schd_fleet as fleet,
    Frst_Flt_Ct as frst_flt_ind,
    pax_vendor as vendor,
    D0_Ct as num,
    DOT_OA_Sub_Op_Ct as den
from 
    zods_kpi.flp_dtl_m
where 
    data_cut = 'STANDARD'
    and DOT_OA_Sub_Op_Ct = 1
    and flt_orig_dt between '2025-06-01' and '2026-05-30'
"""

# --- Execute ---
with teradatasql.connect(host=TD_HOST, user=TD_USER, password=TD_PASS, logmech=TD_LOGMECH) as conn:
    df = pd.read_sql(query, conn)

df.to_parquet('teradata_cache.parquet')
print(f'Saved {len(df):,} rows to teradata_cache.parquet')
print(f'Columns: {list(df.columns)}')
df.head()

### Verify cache (if skipping Step 1)

Run this to confirm the parquet cache exists and inspect its contents.

In [ ]:
import pandas as pd
from pathlib import Path

cache = Path('teradata_cache.parquet')
if cache.exists():
    df = pd.read_parquet(cache)
    print(f'Cache exists: {len(df):,} rows, {len(df.columns)} columns')
    print(f'Columns: {list(df.columns)}')
    print(f'\nDimension columns (available for hierarchy):')
    reserved = {'num', 'den', 'Yr_Nb'}
    dims = [c for c in df.columns if c not in reserved]
    print(f'  {dims}')
    print(f'\nNum range: {df["num"].min()} - {df["num"].max()}')
    print(f'Den range: {df["den"].min()} - {df["den"].max()}')
    df.head()
else:
    print('ERROR: teradata_cache.parquet not found. Run Step 1 first.')

---

## Step 2: Design a Hierarchy

**Option A - GUI Editor (requires display):**
```bash
python editor.py
```
This opens a Tkinter window where you can visually build the hierarchy.
On Save, it automatically runs cascade and outputs to `output/`.

> Note: `editor.py` requires a graphical display. It will NOT work on headless
> environments like Red Hat DevSpaces. Use Option B or C instead.

**Option B - Edit JSON directly:**

Create a file in `hierarchies/` with the structure below.
Use the dimension columns from Step 1 as `column` and `category` values.

**Option C - Build in the cell below:**

In [ ]:
import json
from pathlib import Path

# --- Define your hierarchy ---
# Each node: {"column": "<df_column>", "category": "<label>", "children": [...]}
# Add "split": true to create a split branch (alternate dimensional view).
# Add "transform": {"type": "int_flag", "map": {"1": "Label", "0": "Label"}}
#   for columns that need value mapping.

hierarchy = {
    'levels': [
        {
            'column': 'sys',
            'category': 'sys',
            'children': [
                {
                    'column': 'ml_dc_1',
                    'category': 'ml_dc_1',
                    'children': [
                        {
                            'column': 'ml_dc_2',
                            'category': 'ml_dc_2',
                            'children': [
                                {
                                    'column': 'Mo_Nb',
                                    'category': 'Mo_Nb',
                                    'children': [
                                        {
                                            'column': 'dom_int',
                                            'category': 'dom_int',
                                            'children': [
                                                {'column': 'station', 'category': 'station'}
                                            ]
                                        },
                                        {
                                            'column': 'fleet',
                                            'category': 'fleet',
                                            'split': True
                                        }
                                    ]
                                }
                            ]
                        }
                    ]
                }
            ]
        }
    ]
}

# --- Save ---
HIERARCHY_NAME = 'My D0'  # <-- Change this name

hier_dir = Path('hierarchies')
hier_dir.mkdir(exist_ok=True)
hier_path = hier_dir / f'{HIERARCHY_NAME}.json'

with open(hier_path, 'w', encoding='utf-8') as f:
    json.dump(hierarchy, f, indent=4)

print(f'Saved hierarchy to: {hier_path}')
print(f'\nStructure:')
print(json.dumps(hierarchy, indent=2)[:600])

---

## Step 3: Define Goals

The `goals.csv` file defines explicit targets for anchor nodes.
The cascade distributes stretch from these anchors to all descendants.

**Format:**
```
KPI_Name,Node_Name,Baseline,Baseline_Num,Baseline_Den,Stretch,KPI_Num,KPI_Den,Goal
```

- `Node_Name`: format is `category;name` (e.g. `system;sys`, `carrier;DL`)
- Node matching uses name fallback, so `carrier;DL` will match a node named `DL` even if the tree uses `ml_dc_1` as the category.

Edit `goals.csv` directly or run the cell below to inspect/modify it.

In [ ]:
import pandas as pd

goals = pd.read_csv('goals.csv')
print('Current goals:')
goals

In [ ]:
# --- Uncomment and edit to update goals programmatically ---

# import pandas as pd
# goals = pd.DataFrame([
#     {'KPI_Name': 'D0', 'Node_Name': 'system;sys', 'Baseline': 68.29,
#      'Baseline_Num': 1216960, 'Baseline_Den': 1782050, 'Stretch': '5%',
#      'KPI_Num': 1277808, 'KPI_Den': 1782050, 'Goal': 71.70},
#     {'KPI_Name': 'D0', 'Node_Name': 'carrier;ML', 'Baseline': 66.24,
#      'Baseline_Num': 769980, 'Baseline_Den': 1162327, 'Stretch': '5%',
#      'KPI_Num': 808479, 'KPI_Den': 1162327, 'Goal': 69.56},
#     {'KPI_Name': 'D0', 'Node_Name': 'carrier;DC', 'Baseline': 72.13,
#      'Baseline_Num': 446980, 'Baseline_Den': 619723, 'Stretch': '5%',
#      'KPI_Num': 469329, 'KPI_Den': 619723, 'Goal': 75.73},
# ])
# goals.to_csv('goals.csv', index=False)
# print('Updated goals.csv')
# goals

---

## Step 4: Run the Cascade

This builds the tree, applies goals, cascades stretch to all descendants,
and writes the output JSON + CSV.

Set `HIERARCHY_NAME` to match the name from Step 2.

In [ ]:
from pathlib import Path
from cascade import cascade

HIERARCHY_NAME = 'My D0'  # <-- Must match the name from Step 2

hierarchy_file = Path(f'hierarchies/{HIERARCHY_NAME}.json')
output_file = Path(f'output/{HIERARCHY_NAME}.json')

if not hierarchy_file.exists():
    print(f'ERROR: {hierarchy_file} not found. Run Step 2 first.')
else:
    tree = cascade(
        goals_file=Path('goals.csv'),
        hierarchy_file=hierarchy_file,
        cache_file=Path('teradata_cache.parquet'),
        output_file=output_file
    )
    print(f'\nDone! Outputs:')
    print(f'  Tree JSON: {output_file}')
    print(f'  Cascaded CSV: output/{HIERARCHY_NAME}_cascaded.csv')

### Inspect the cascaded output

In [ ]:
import pandas as pd

HIERARCHY_NAME = 'My D0'  # <-- Same as above

csv_path = f'output/{HIERARCHY_NAME}_cascaded.csv'
df = pd.read_csv(csv_path)

print(f'Cascaded output: {len(df)} rows')
print(f'\nSource breakdown:')
print(df['Source'].value_counts().to_string())
print(f'\nTier 1-3 summary:')
df[df['Tier'] <= 3]

---

## Step 5: Launch the Viewer

Copy and paste the command below into your terminal.

**On Red Hat DevSpaces:** Watch for the "Open in New Tab" notification after running.

In [ ]:
HIERARCHY_NAME = 'My D0'

print('Run this in your terminal:')
print()
print('  streamlit run app.py --server.address 0.0.0.0 --server.port 8501 --server.headless true')
print()
print('Then:')
print('  - Local: open http://localhost:8501')
print('  - DevSpaces: click "Open in New Tab" when the port notification appears')
print(f'  - Select "{HIERARCHY_NAME}" from the KPI dropdown in the sidebar')

---

## Configuration Reference

### Cascade Basis

Controls how both contributions and stretch are distributed. Edit `CASCADE_BASIS` in `config.py`:

| Value | Distribution | Effect |
|-------|-------------|--------|
| `"num"` (default) | `child.num / parent.num` | Uniform percentage improvement across children |
| `"den"` | `child.den / parent.den` | Volume-proportional distribution |

### Value Transforms

To map raw values to display labels, add entries to `TRANSFORMS` in `editor.py`:

```python
TRANSFORMS = {
    "frst_flt_ind": {"type": "int_flag", "map": {"1": "First Flight", "0": "Not First Flight"}},
    "dom_int": {"type": "map", "map": {"D": "Domestic", "I": "International"}},
}
```

### Adding Dimensions

1. Add column to the SQL query in Step 1
2. Re-run Step 1 to refresh the parquet
3. The new column appears automatically in the hierarchy editor